## Ex 0 - 1. Upload data files to Databricks


### Strucure for catalog, table and volume
 a) Create a new catalog, airbnb

  b) Create a schema, hosts

  c) Create a volume, csv_files and upload the csv file from Kaggle here

### ETL, SDP (Structured Data Processing)

  d) Now create an ETL pipeline for SDP. This pipeline should be able to produce a streaming table under the same schema created in question b) above. Use pyspark for this.

  e) Recreate d) but using SQL isntead of pyspark.

In [0]:
from pyspark import pipelines as dp 

# BASE PATH to the csv - data file 
BASE_PATH = "Volumes/airbnb/hosts/csv_files"


# parse schema from csv file and use it for the streaming table
# as readStream processes data continously and can't look ahead of time to infer schema

schema = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(f"{BASE_PATH}/Airbnb_Open_Data.csv")
    .schema
)

@dp.table(
    name="airbnb_hosts",
    comment="Raw orders data as the bronze layer in medallion architecture",
    table_properties={
        "delta.columnMapping.mode": "name",
        "delta.minReaderVersion": "2",
        "delta.minWriterVersion": "5"
    }
)

def airbnb_hosts_stream():
    return(
        spark.readStream.format("csv")
        .schema(schema)
        .options(header="true",inferSchema="true", encoding="UTF-8")
        .schema(schema)
        .load(f"{BASE_PATH}/Airbnb_Open_Data.csv")
    )